# 07 — SAMap gene-level conservation and divergence (all-samples vanilla integration)

**Source: `2026_08_31_samap_vanilla_fibdec_joint.h5ad`** — the all-samples vanilla integration
(90,730 cells x 70,299 genes; 36,601 human + 33,698 mouse). It carries *both* gene-gene graphs:

| slot | what | scale | nnz (total / cross-species) |
|---|---|---|---|
| `varp["homology_graph"]` | initial, **sequence**-derived | BLAST/phmmer bit scores, 28.3 – 17,570 (median 241) | 1,508,062 / **754,031** |
| `varp["homology_graph_reweighted"]` | after SAMap's **expression** refinement | correlation weights, 9.0e-7 – 0.892 (median 0.055) | 451,106 / **225,553** |

This is the conservation/divergence signal directly: a candidate edge exists because two genes look
alike in *sequence*; it survives refinement because the two genes also behave alike in *expression*.

**Two facts that determine how this must be analysed — both verified below, neither assumed:**

1. **Refinement only prunes and rescales; it never adds.** Reweighted-only edges = **0**. So the
   sequence graph is the candidate universe and the refined graph is a subset of it.
2. **The two graphs are on incommensurable scales.** Bit scores vs correlation weights. A raw
   ratio `reweighted/initial` is therefore **meaningless** — it comes out < 0.5 for 100% of edges
   purely from the unit change, which says nothing biological. This notebook does not use it, and
   section 2.1 demonstrates why so nobody reintroduces it.

The two valid readouts are **survival** (binary, scale-free) and **rank within a gene's partners**.

In [1]:
# --- paths & helpers ----------------------------------------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
SAMR = REV/"samap_integrate"
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
DATA = pathlib.Path("data")          # staged artifacts
MIS  = X/"results/mi_stratified"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    try:
        ok = (abs(float(got)-float(expected)) <= tol*max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    print(f"\n{len(d)} checks | {int((d.status.str.strip()=='MISMATCH').sum())} MISMATCH")
    return d

print("data dir:", DATA.resolve() if DATA.exists() else "MISSING",
      "| files:", len(list(DATA.glob("*"))) if DATA.exists() else 0)


data dir: /n/lab_storage/eddy_lab/users/nhilgert/xmens_final/notebooks/verify/data | files: 29


## 1. Load both graphs and confirm the structural facts

In [2]:
import anndata as ad, scipy.sparse as sp
A = ad.read_h5ad(NET/"2026_08_31_samap_vanilla_fibdec_joint.h5ad")
G0 = sp.csr_matrix(A.varp["homology_graph"])           # sequence
G1 = sp.csr_matrix(A.varp["homology_graph_reweighted"])# expression-refined
gn = np.asarray(A.var_names, dtype=str)
is_hs = np.char.startswith(gn, "hs_"); is_mm = np.char.startswith(gn, "mm_")
print("cells x genes:", A.shape, "| human genes:", is_hs.sum(), "| mouse genes:", is_mm.sum())
verify("total genes", int(A.shape[1]), 70299, tol=0)
verify("human genes", int(is_hs.sum()), 36601, tol=0)
verify("mouse genes", int(is_mm.sum()), 33698, tol=0)
verify("sequence graph nnz", int(G0.nnz), 1508062, tol=0)
verify("refined graph nnz", int(G1.nnz), 451106, tol=0)
verify("no genes outside hs_/mm_", int((~is_hs & ~is_mm).sum()), 0, tol=0)

C0 = G0[np.ix_(is_hs, is_mm)].tocoo(); C1 = G1[np.ix_(is_hs, is_mm)].tocoo()
verify("candidate cross-species edges", int(C0.nnz), 754031, tol=0)
verify("surviving cross-species edges", int(C1.nnz), 225553, tol=0)
E0 = set(zip(C0.row.tolist(), C0.col.tolist())); E1 = set(zip(C1.row.tolist(), C1.col.tolist()))
added = E1 - E0
print(f"\nedges: candidate={len(E0):,} surviving={len(E1):,} pruned={len(E0-E1):,} added={len(added):,}")
verify("refinement ADDS no edges", len(added), 0, tol=0,
       note="the sequence graph is the candidate universe; refinement is a subset operation")

cells x genes: (90730, 70299) | human genes: 36601 | mouse genes: 33698
[PASS ] total genes: recomputed=70299  recorded=70299
[PASS ] human genes: recomputed=36601  recorded=36601
[PASS ] mouse genes: recomputed=33698  recorded=33698
[PASS ] sequence graph nnz: recomputed=1508062  recorded=1508062
[PASS ] refined graph nnz: recomputed=451106  recorded=451106
[PASS ] no genes outside hs_/mm_: recomputed=0  recorded=0
[PASS ] candidate cross-species edges: recomputed=754031  recorded=754031
[PASS ] surviving cross-species edges: recomputed=225553  recorded=225553



edges: candidate=754,031 surviving=225,553 pruned=528,478 added=0
[PASS ] refinement ADDS no edges: recomputed=0  recorded=0   (the sequence graph is the candidate universe; refinement is a subset operation)


True

## 2. Readout 1 — survival: how much of sequence similarity is *not* corroborated by expression

The headline divergence number. Every cross-species candidate edge is a hypothesis that two genes
correspond; refinement tests it against expression.

In [3]:
surv = len(E1)/len(E0)
print(f"cross-species candidate edges: {len(E0):,}")
print(f"  survive refinement : {len(E1):,}  ({100*surv:.1f}%)   <- expression corroborates sequence")
print(f"  pruned to zero     : {len(E0-E1):,}  ({100*(1-surv):.1f}%)   <- sequence-similar, expression-divergent")
verify("fraction of candidate edges surviving refinement", round(surv,4), 0.2992, tol=0.01)
verify("majority of sequence-similar pairs are pruned", bool(surv < 0.5), True,
       note="70.1% of sequence-based cross-species gene links are not supported by expression")

# per-gene degree: how many partners does a gene start with vs keep?
d0 = np.asarray((C0 > 0).sum(1)).ravel(); d1 = np.asarray((C1 > 0).sum(1)).ravel()
hs_names = np.char.replace(gn[is_hs], "hs_", "")
deg = pd.DataFrame(dict(human=hs_names, n_candidate=d0, n_surviving=d1))
deg = deg[deg.n_candidate > 0]
print(f"\nhuman genes with >=1 candidate partner: {len(deg):,}")
print(f"  median candidate partners per gene: {deg.n_candidate.median():.0f}")
print(f"  median surviving partners per gene : {deg.n_surviving.median():.0f}")
print(f"  genes losing ALL partners          : {int((deg.n_surviving==0).sum()):,}"
      f" ({100*(deg.n_surviving==0).mean():.1f}%)")
verify("candidate graph is many-to-many (median >1 partner per gene)",
       bool(deg.n_candidate.median() > 1), True,
       note="this is why strict 1:1 orthology and the SAMap graph disagree")

cross-species candidate edges: 754,031
  survive refinement : 225,553  (29.9%)   <- expression corroborates sequence
  pruned to zero     : 528,478  (70.1%)   <- sequence-similar, expression-divergent
[PASS ] fraction of candidate edges surviving refinement: recomputed=0.2991  recorded=0.2992
[PASS ] majority of sequence-similar pairs are pruned: recomputed=True  recorded=True   (70.1% of sequence-based cross-species gene links are not supported by expression)

human genes with >=1 candidate partner: 18,493
  median candidate partners per gene: 2
  median surviving partners per gene : 1
  genes losing ALL partners          : 3,003 (16.2%)
[PASS ] candidate graph is many-to-many (median >1 partner per gene): recomputed=True  recorded=True   (this is why strict 1:1 orthology and the SAMap graph disagree)


True

### 2.1 Why the weight *ratio* must not be used

Demonstration, not assertion. The ratio `reweighted/initial` is < 0.5 for **100%** of shared edges
and > 1 for **0%** — an artifact of bit scores (median 241) versus correlation weights (median
0.055), i.e. a ~4400x unit difference. It carries no information about conservation. Any future
analysis wanting a graded measure should rank *within* a gene's partner set (section 3), not divide.

In [4]:
K0 = dict(zip(zip(C0.row.tolist(), C0.col.tolist()), C0.data.tolist()))
K1 = dict(zip(zip(C1.row.tolist(), C1.col.tolist()), C1.data.tolist()))
shared = list(E1)
ratio = np.array([K1[k]/K0[k] for k in shared if K0[k] > 0])
print(f"weight ratio on {len(ratio):,} shared edges:")
print(f"  median={np.median(ratio):.3g}  frac>1={float((ratio>1).mean()):.4f}  frac<0.5={float((ratio<0.5).mean()):.4f}")
print(f"  initial  weights: median={np.median(C0.data):.1f}  (bit scores)")
print(f"  refined  weights: median={np.median(C1.data):.4f}  (correlation)")
verify("ratio > 1 for essentially no edges", round(float((ratio>1).mean()),4), 0.0, tol=0.001,
       note="pure unit artifact - DO NOT interpret this ratio as divergence")
verify("scales differ by >1000x", bool(np.median(C0.data)/np.median(C1.data) > 1000), True)

weight ratio on 225,553 shared edges:
  median=0.000135  frac>1=0.0000  frac<0.5=1.0000
  initial  weights: median=241.2  (bit scores)
  refined  weights: median=0.0550  (correlation)
[PASS ] ratio > 1 for essentially no edges: recomputed=0.0  recorded=0.0   (pure unit artifact - DO NOT interpret this ratio as divergence)
[PASS ] scales differ by >1000x: recomputed=True  recorded=True


True

## 3. Readout 2 — is the 1:1 ortholog a gene's top partner?

The direct, all-samples measurement of paralog substitution, and the sharpest thing this object can
tell us. For each human gene that (a) has a strict 1:1 mouse ortholog and (b) retains at least one
surviving partner: **does SAMap's refined graph rank the ortholog first?**

- ortholog ranked first → **conserved**: sequence and expression agree on the partner
- a paralog ranked first → **substitution**: the functional partner is not the ortholog
- ortholog pruned entirely → **ortholog rejected**: expression contradicts the 1:1 link

In [5]:
# The Ensembl table is LONG format: one row per species per component.
#   component_id | species | ensembl_gene_id | gene_symbol | biotype | homology_type | ...
# It also has human_copy_number / mouse_copy_number columns, so naive "column containing 'human'"
# matching silently picks COPY NUMBERS and yields garbage pairs. Pivot on component_id instead.
orth = None
for cand in (DATA/"ensembl_human_mouse_components.tsv",
             pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/"
                          "ensembl_human_mouse_components.tsv")):
    if cand.exists():
        orth = pd.read_csv(cand, sep="\t"); print("ortholog table:", cand); break

if orth is None:
    print("ortholog table NOT FOUND - sections 3-5 skipped")
    pairs = {}
else:
    print("cols:", list(orth.columns))
    assert {"component_id","species","gene_symbol","homology_type"} <= set(orth.columns), \
        f"unexpected schema: {list(orth.columns)}"
    o2o = orth[orth.homology_type == "ortholog_one2one"]
    w = (o2o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                         aggfunc="first")
            .dropna(subset=["human","mouse"]))
    pairs = dict(zip(w["human"].astype(str).str.upper(), w["mouse"].astype(str)))
    print(f"strict 1:1 components: {len(w):,} -> usable symbol pairs: {len(pairs):,}")
    print("  e.g.", list(pairs.items())[:4])
    verify("1:1 pairs are symbol->symbol, not numbers",
           all(not str(v).isdigit() for v in list(pairs.values())[:200]), True,
           note="guards the long-format pivot trap")
    verify("known ortholog TSPAN6->Tspan6 present", pairs.get("TSPAN6"), "Tspan6", tol=0)

ortholog table: data/ensembl_human_mouse_components.tsv
cols: ['component_id', 'species', 'ensembl_gene_id', 'gene_symbol', 'biotype', 'homology_type', 'gene_tree_id', 'human_copy_number', 'mouse_copy_number', 'source_release']
strict 1:1 components: 16,277 -> usable symbol pairs: 16,277
  e.g. [('TSPAN6', 'Tspan6'), ('TNMD', 'Tnmd'), ('DPM1', 'Dpm1'), ('SCYL3', 'Scyl3')]
[PASS ] 1:1 pairs are symbol->symbol, not numbers: recomputed=True  recorded=True   (guards the long-format pivot trap)
[PASS ] known ortholog TSPAN6->Tspan6 present: recomputed=Tspan6  recorded=Tspan6


In [6]:
if orth is not None:
    mm_names = np.char.replace(gn[is_mm], "mm_", "")
    hs_idx = {g.upper(): i for i, g in enumerate(hs_names)}
    mm_idx = {g: j for j, g in enumerate(mm_names)}
    C1r = C1.tocsr(); C0r = C0.tocsr()
    rows = []
    for h, m in pairs.items():
        i = hs_idx.get(h); j = mm_idx.get(m)
        if i is None or j is None: continue
        s, e = C1r.indptr[i], C1r.indptr[i+1]
        if e == s: continue                       # no surviving partners at all
        cols, vals = C1r.indices[s:e], C1r.data[s:e]
        order = np.argsort(-vals)
        ranked = cols[order]
        top_mm = mm_names[ranked[0]]
        orth_pos = int(np.where(ranked == j)[0][0]) + 1 if j in set(ranked.tolist()) else None
        rows.append(dict(human=h, ortholog=m, n_surviving=len(cols), top_partner=top_mm,
                         ortholog_rank=orth_pos,
                         verdict=("CONSERVED" if top_mm == m else
                                  ("SUBSTITUTED" if orth_pos is not None else "ORTHOLOG_REJECTED"))))
    R = pd.DataFrame(rows)
    print(f"human genes assessable (1:1 ortholog + >=1 surviving partner): {len(R):,}\n")
    vc = R.verdict.value_counts()
    for k in ("CONSERVED","SUBSTITUTED","ORTHOLOG_REJECTED"):
        if k in vc: print(f"  {k:18s} {vc[k]:6,d}  ({100*vc[k]/len(R):5.1f}%)")
    R.to_csv("samap_ortholog_verdicts.csv", index=False)
    verify("verdict classes are exhaustive", int(vc.sum()), len(R), tol=0)
    verify("a majority of assessable genes keep the ortholog as top partner",
           bool(vc.get("CONSERVED",0) > len(R)/2), None,
           note=f"CONSERVED={vc.get('CONSERVED',0)} of {len(R)}")

human genes assessable (1:1 ortholog + >=1 surviving partner): 13,649

  CONSERVED           8,425  ( 61.7%)
  SUBSTITUTED         3,196  ( 23.4%)
  ORTHOLOG_REJECTED   2,028  ( 14.9%)
[PASS ] verdict classes are exhaustive: recomputed=13649  recorded=13649
[n/a  ] a majority of assessable genes keep the ortholog as top partner: recomputed=True  recorded=None   (CONSERVED=8425 of 13649)


## 4. Named divergence cases — cross-checked against the independent GenePairFinder analysis

The substitutions the graph itself identifies, intersected with the recurrent paralog pairs found
independently by `GenePairFinder` across the 12 per-pairing runs (`genepairs_v3_all.csv`). Two
different instruments on two different object sets; agreement is meaningful.

In [7]:
if orth is not None and (DATA/"genepairs_v3_all.csv").exists():
    D = pd.read_csv(DATA/"genepairs_v3_all.csv")
    Cg = D[(~D.housekeeping) & (~D.promiscuous) & (~D.is_1to1)]
    gpf = (Cg.groupby(["human","mouse"]).run.nunique().reset_index(name="n_runs")
             .query("n_runs >= 8"))
    sub = R[R.verdict == "SUBSTITUTED"][["human","ortholog","top_partner","ortholog_rank","n_surviving"]]
    both = sub.merge(gpf, left_on=["human","top_partner"], right_on=["human","mouse"], how="inner")
    print(f"substitutions named by BOTH the refined graph and GenePairFinder: {len(both)}\n")
    for _, r in both.sort_values("n_runs", ascending=False).head(25).iterrows():
        print(f"   {r.human:>12s}: ortholog {r.ortholog:<12s} (rank {r.ortholog_rank}) "
              f"-> top partner {r.top_partner:<12s}  GPF {int(r.n_runs)}/12")
    both.to_csv("samap_substitutions_cross_confirmed.csv", index=False)
    verify("independent methods agree on >=1 substitution", bool(len(both) >= 1), True,
           note="refined graph (all-samples) vs GenePairFinder (12 per-pairing runs)")
else:
    print("prerequisites missing - section 4 skipped")

substitutions named by BOTH the refined graph and GenePairFinder: 41

          ITGAV: ortholog Itgav        (rank 2.0) -> top partner Itga5         GPF 12/12
           FAT4: ortholog Fat4         (rank 2.0) -> top partner Pcdh9         GPF 12/12
           LRBA: ortholog Lrba         (rank 3.0) -> top partner Nbea          GPF 12/12
          PCDH7: ortholog Pcdh7        (rank 5.0) -> top partner Pcdh9         GPF 11/12
          LSAMP: ortholog Lsamp        (rank 2.0) -> top partner Negr1         GPF 11/12
            PGR: ortholog Pgr          (rank 3.0) -> top partner Ar            GPF 10/12
        HNRNPH3: ortholog Hnrnph3      (rank 4.0) -> top partner Hnrnpf        GPF 10/12
          HIF1A: ortholog Hif1a        (rank 2.0) -> top partner Epas1         GPF 10/12
           NFIX: ortholog Nfix         (rank 2.0) -> top partner Nfia          GPF 10/12
          BMP2K: ortholog Bmp2k        (rank 2.0) -> top partner Aak1          GPF 10/12
         TXNRD1: ortholog Txnrd1       (

## 5. Do conserved-positional genes keep their orthologs more often?

Joins the verdicts to the transcriptome-wide positional-conservation ranking. A positive result
would say the genes carrying conserved spatial information are also the genes whose 1:1 ortholog
relationship survives expression refinement — two independent notions of conservation agreeing.

In [8]:
rank_p = next((p for p in (DATA/"conservation_ranking_all_genes.tsv",
                           X/"results/conservation/conservation_ranking_all_genes.tsv") if p.exists()), None)
if orth is not None and rank_p is not None:
    K = pd.read_csv(rank_p, sep="\t")
    gcol = next((c for c in K.columns if c.lower() in ("human","gene","symbol")), K.columns[0])
    K["_g"] = K[gcol].astype(str).str.upper()
    J = R.merge(K, left_on="human", right_on="_g", how="inner")
    print(f"joined genes: {len(J):,} | verdict x conservation class")
    if "verdict_y" in J.columns or "verdict" in K.columns:
        vcol = "verdict_y" if "verdict_y" in J.columns else "verdict"
        ct = pd.crosstab(J["verdict_x" if "verdict_x" in J.columns else "verdict"], J[vcol], normalize="index")
        print(ct.round(3).to_string())
    for pc in ("pct_human","pct_mouse"):
        if pc in J.columns:
            g = J.groupby("verdict_x" if "verdict_x" in J.columns else "verdict")[pc].median()
            print(f"\n  median {pc} by graph verdict:\n{g.round(3).to_string()}")
else:
    print("conservation ranking not staged - section 5 skipped")

joined genes: 9,165 | verdict x conservation class
verdict_y          CONSERVED_POSITIONAL  CONSERVED_UNCLEAR_SHAPE  HUMAN_ONLY  INVERTED  MOUSE_ONLY  NOT_POSITIONAL
verdict_x                                                                                                         
CONSERVED                         0.029                    0.017       0.075     0.008       0.081           0.791
ORTHOLOG_REJECTED                 0.009                    0.019       0.098     0.034       0.060           0.780
SUBSTITUTED                       0.014                    0.016       0.093     0.012       0.059           0.806

  median pct_human by graph verdict:
verdict_x
CONSERVED            0.611
ORTHOLOG_REJECTED    0.530
SUBSTITUTED          0.577

  median pct_mouse by graph verdict:
verdict_x
CONSERVED            0.619
ORTHOLOG_REJECTED    0.451
SUBSTITUTED          0.562


## 6. Limits

1. **`uns["gnnm_corr"]` is on a different gene space** (39,843 x 39,843) than the `varp` graphs
   (70,299) — presumably SAMap's post-filter gene set. Not used here; joining it would need its own
   index, which the export does not carry.
2. **One integration, one homology model.** These graphs are the vanilla arm built on the phmmer
   map. The harmony export carries byte-identical `varp` slots (the arms differ in cell-level
   structure, not the gene graph), so this analysis cannot distinguish them — and does not try to.
3. **Survival is binary by construction.** SAMap prunes at its own internal threshold; a pair just
   above it and one far above it both count as surviving. Section 3's rank-within-gene is the
   graded refinement, and it is a *relative* measure only.
4. **The all-samples pickles remain unreadable** (`KeyError: '_adata_ref'` in the SAMap venv —
   confirmed, not inferred), so `GenePairFinder` cannot be rerun on the all-samples integration.
   Section 4's cross-check therefore compares all-samples graph structure against per-pairing
   driver lists, which is a genuine independence but not a like-for-like replication.

## Summary

In [9]:
check_summary()


19 checks | 0 MISMATCH


,claim,recomputed,recorded,status
0,total genes,70299,70299,PASS
1,human genes,36601,36601,PASS
2,mouse genes,33698,33698,PASS
3,sequence graph nnz,1508062,1508062,PASS
4,refined graph nnz,451106,451106,PASS
5,no genes outside hs_/mm_,0,0,PASS
6,candidate cross-species edges,754031,754031,PASS
7,surviving cross-species edges,225553,225553,PASS
8,refinement ADDS no edges,0,0,PASS
9,fraction of candidate edges surviving refinement,0.2991,0.2992,PASS
